<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Neutrino012.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from itertools import product
from multiprocessing import Pool

# -------------------------------
# HES-inspired mass model
# -------------------------------
def hes_mass_model(params):
    delta, kappa = params
    base = 1.0
    small_split = delta * 0.01
    large_split = kappa * 0.30
    m1 = base
    m2 = base + small_split
    m3 = base + small_split + large_split
    return np.array([m1, m2, m3])

# -------------------------------
# Objective: match Δm² ratio
# -------------------------------
def objective(params):
    masses = hes_mass_model(params)
    m1, m2, m3 = masses
    delta_m2_21 = m2**2 - m1**2
    delta_m2_32 = m3**2 - m2**2
    if delta_m2_21 <= 0 or delta_m2_32 <= 0:
        return np.inf
    ratio = delta_m2_32 / delta_m2_21
    target_ratio = 33.3
    return abs(ratio - target_ratio)

# -------------------------------
# Grid generation and refinement
# -------------------------------
def generate_grid(bounds, n_points, scale='linear'):
    grids = []
    for (lo, hi) in bounds:
        if scale == 'log':
            grids.append(np.exp(np.linspace(np.log(lo), np.log(hi), n_points)))
        else:
            grids.append(np.linspace(lo, hi, n_points))
    return list(product(*grids))

def hierarchical_sweep(
    initial_bounds,
    levels=4,
    points_per_dim=8,
    top_k=6,
    refine_factor=0.4,
    scale='linear',
    parallel=False
):
    current_regions = [(initial_bounds, None)]
    best_candidates = []

    for level in range(levels):
        print(f"Level {level+1}/{levels}, {len(current_regions)} region(s)")
        region_results = []

        for bounds, _ in current_regions:
            grid = generate_grid(bounds, points_per_dim, scale=scale)
            if parallel:
                with Pool() as pool:
                    scores = pool.map(objective, grid)
            else:
                scores = list(map(objective, grid))

            scored = list(zip(scores, grid))
            scored.sort(key=lambda x: x[0])
            region_results.extend(scored[:top_k])

        region_results.sort(key=lambda x: x[0])
        best_candidates = region_results[:top_k]
        print("  Best scores this level:", [round(s, 6) for s, _ in best_candidates])

        next_regions = []
        for score, params in best_candidates:
            spans = [(hi - lo) * (refine_factor ** level) for (lo, hi), p in zip(initial_bounds, params)]
            new_bounds = []
            for p, s, (lo0, hi0) in zip(params, spans, initial_bounds):
                lo = max(p - s/2, lo0)
                hi = min(p + s/2, hi0)
                if lo >= hi:
                    eps = 1e-12
                    lo = max(lo - eps, lo0)
                    hi = min(hi + eps, hi0)
                new_bounds.append((lo, hi))
            next_regions.append((new_bounds, params))
        current_regions = next_regions

    return best_candidates

# -------------------------------
# Run the sweep
# -------------------------------
initial_bounds = [(0.001, 1.0), (0.001, 1.0)]  # δ and κ
best = hierarchical_sweep(
    initial_bounds,
    levels=4,
    points_per_dim=8,
    top_k=6,
    refine_factor=0.4,
    scale='linear',
    parallel=False
)

# -------------------------------
# Display results with physical scaling
# -------------------------------
print("\n🎯 Top results:")
scale_factor = 0.01  # Adjust based on HES calibration

for score, params in best:
    masses = hes_mass_model(params)
    m1, m2, m3 = masses
    delta_m2_21 = m2**2 - m1**2
    delta_m2_32 = m3**2 - m2**2
    ratio = delta_m2_32 / delta_m2_21
    physical_masses = np.array(masses) * scale_factor

    print(f"score={score:.6f}, δ={params[0]:.6f}, κ={params[1]:.6f}")
    print(f"  masses (sim units): {np.round(masses, 6)}")
    print(f"  Δm²_21={delta_m2_21:.6e}, Δm²_32={delta_m2_32:.6e}, ratio={ratio:.2f}")
    print(f"  masses (eV): {np.round(physical_masses, 6)}\n")


Level 1/4, 1 region(s)
  Best scores this level: [np.float64(0.010928), np.float64(0.648446), np.float64(0.669364), np.float64(1.308758), np.float64(1.326866), np.float64(1.970012)]
Level 2/4, 6 region(s)
  Best scores this level: [np.float64(0.010928), np.float64(0.05799), np.float64(0.105047), np.float64(0.177369), np.float64(0.224455), np.float64(0.224455)]
Level 3/4, 6 region(s)
  Best scores this level: [np.float64(0.026726), np.float64(0.045553), np.float64(0.073796), np.float64(0.092626), np.float64(0.092626), np.float64(0.120872)]
Level 4/4, 6 region(s)
  Best scores this level: [np.float64(0.007162), np.float64(0.021077), np.float64(0.025988), np.float64(0.037283), np.float64(0.039905), np.float64(0.039905)]

🎯 Top results:
score=0.007162, δ=0.713756, κ=0.713756
  masses (sim units): [1.       1.007138 1.221264]
  Δm²_21=1.432606e-02, Δm²_32=4.771605e-01, ratio=33.31
  masses (eV): [0.01     0.010071 0.012213]

score=0.021077, δ=0.707640, κ=0.707640
  masses (sim units): [1.  